### Troubleshooting notes for future reference
Identifies the location of the CA bundle used by the certifi package, used to verify SSL certificates in Python applications if error in SSL verification occurs.

```python
import certifi
certifi.where()
```

after locating the file location, run the following commands in terminal to update the CA bundle

```python
setx SSL_CERT_FILE "path\to\cacert.pem"
setx REQUESTS_CA_BUNDLE "path\to\cacert.pem"
```

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import xarray as xr
from dask.distributed import Client, LocalCluster
from IPython.display import Image
from odc.stac import configure_rio, stac_load
from shapely.geometry import box

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [ ]:
cluster = LocalCluster(
    processes=True,
    n_workers=8,
    threads_per_worker=2,
    memory_limit="6GB",
)
client = Client(cluster)
configure_rio(cloud_defaults=True, client=client)
client

In [ ]:
# check available landsat collections within the catalog
all_collections = [i.id for i in catalog.get_collections()]
landsat_collections = [
    collection for collection in all_collections if "landsat" in collection
]
landsat_collections

In [ ]:
# identify bbox for singapore, date range, and collection
bbox = [103.577522, 1.160000, 104.107269, 1.480106]
dates = "2025-01-01/2025-12-31"
collection = "landsat-c2-l2"
cloudylvl = 50

In [ ]:
# create a search object with the parameters
search = catalog.search(
    collections=[collection],
    bbox=bbox,
    datetime=dates,
    query={"eo:cloud_cover": {"lt": cloudylvl}},
)

# view items from the search
items = search.item_collection()
print(f"Returned {len(items)} Items:")
item_id = {(i, item.id): i for i, item in enumerate(items)}
item_id

In [ ]:
import geopandas as gpd

In [ ]:
# load searched item information into a dataframe
# converts items to a method, then creates features from it
images = gpd.GeoDataFrame.from_features(items.to_dict(), crs="epsg:4326")
images.head()

In [ ]:
images.dtypes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
# plot cloud cover percentage over time from images
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(pd.to_datetime(images["datetime"], utc=True), images["eo:cloud_cover"])

# set date format and rotate xaxis labels 45 degrees
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%Y'))
plt.xticks(rotation=25)

# set titles
plt.title("Cloud Cover % over Time")
plt.xlabel("Date")
plt.ylabel("Cloud Cover %")
plt.show()

In [ ]:
# select the image with the lowest cloud percent
low_cloud = min(items, key=lambda item: item.properties["eo:cloud_cover"])

In [ ]:
from IPython.display import Image

Every image retrieved from the database has a full-color "rendered_preview" image that has an associated web link to retrieve it. The Image function takes a web url and creates a python object that can be visualized as an output.

In [ ]:
Image(url=low_cloud.assets['rendered_preview'].href, width=500)

In [ ]:
import mapclassify

In [ ]:
import shapely.geometry as geom

In [ ]:
# convert bbox to a geopandas geoseries to plot with matplotlib
bbox_geom = gpd.GeoSeries(geom.box(*bbox), crs='epsg:4326')

To confirm that the images we pulled all cover Singapore's bbox extent, the following code plots the image boundaries on a dynamic map as well as geometries of all landsat images over the bbox extent. These plots confirm that Singapore lies fully within all images retrieved.

In [ ]:
images[["geometry", "datetime", "eo:cloud_cover"]].explore(
    column="eo:cloud_cover", style_kwds={"fillOpacity": 0.05}
)

In [ ]:
# plot geometry of the first image over bbox
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 4))
images.boundary.plot(ax=ax, color="grey", lw = 0.5, alpha = 0.25)
bbox_geom.plot(ax=ax, alpha=0.5, color='red')
ax.set_title("Landsat Images over Singapore (2025)")
plt.show()

In [ ]:
images['datetime']

In [ ]:
# fix datetime on the x axis to just show year-month on an angle
images['datetime'] = pd.to_datetime(images['datetime'])
ts = images.set_index("datetime").sort_index()["eo:cloud_cover"]
ts.plot(title="Singapore Cloud Cover Jan-Dec 2025", figsize=(6, 4))
plt.xlabel("Date")
plt.ylabel("Cloud Cover (%)")

In [ ]:
import rich.table

In [ ]:
# create a rich table with all bands/elements of the raster file
table = rich.table.Table("Asset Key", "Descripiption")
for asset_key, asset in low_cloud.assets.items():
    # print(f"{asset_key:<25} - {asset.title}")
    table.add_row(asset_key, asset.title)

table

In [ ]:
# identify bands of interest to load
bandsofinterest = ['red', 'green', 'blue', 'nir08', 'qa_pixel']

In [ ]:
import xarray as xr
import rasterio.features

In [ ]:
# load raster datasets based on parameters
data = stac_load(
    items=items,
    bands=bandsofinterest,
    bbox=bbox,
    chunks={}
)
data

In [ ]:
da = data.to_array(dim="band").compute()
da

In [ ]:
median = da.median(dim='time')
median

In [ ]:
# isolate just red, green, and blue bands in the median data array
test = median.sel(band=['red', 'green', 'blue']).compute()
test

In [ ]:
import xrspatial.multispectral as ms
ms.true_color(*test).plot.imshow()